# 08: Choosing the advanced model

**DATA 5500, Senior Design**

**Sara Choque Molina**

Notebook 07 answered Q1 with one model, TF-IDF with logistic regression. So, now we work on an advanced model, so this notebook runs a search: several candidates on the same labeled articles and the same folds, and a decision at the end.

The candidates and the decision rule are written here, before anything is run, so the winner is not chosen after seeing the numbers.

| | Model | What changes |
|---|---|---|
| 1 | TF-IDF + logistic regression | The reference from notebook 07, not a candidate |
| 2 | TF-IDF + linear SVM | Same word features, a different linear classifier |
| 3 | TF-IDF + XGBoost | Same word features, a tree family instead of a line |
| 4 | TF-IDF + SVD + logistic regression | Word features compressed into topics first |
| 5 | Sentence embeddings + logistic regression | Meaning instead of word counts |

**Decision rule.** Every candidate is scored by Cohen's kappa, averaged over the same 10 random CV splits. The winner is the highest average. A win only counts if the gap over the reference is larger than the spread the reference itself shows across those 10 splits. If it is smaller, the honest conclusion is that the two cannot be told apart at this sample size, and the reference stays.

## Block 1: Load the labeled articles

**Input.** It reads the training set built in notebook 07, so this notebook learns from exactly the same labeled articles.

It is also the first control. Before any model runs, the row count and the label counts are printed, so the notebook shows on its face which file it read and what was in it.

In [1]:
import os

import numpy as np
import pandas as pd

FOLDER = r"C:\Users\sara_\Desktop\College\DATA 5500-01 Senior Des\data bases"

LABELS = ["targeted_threat", "nontargeted_threat", "unrelated"]

training = pd.read_csv(os.path.join(FOLDER, "training_set.csv"))

print("File read:", "training_set.csv")
print("Rows:", len(training))
print("Columns:", list(training.columns))
print()
print("Label counts")
print(training["label"].value_counts().reindex(LABELS).to_string())
print()
print("Rows with no text:", training["text"].isna().sum())
print("Median words per article:", int(training["text"].str.split().str.len().median()))

File read: training_set.csv
Rows: 136
Columns: ['round', 'id', 'url', 'label', 'agreement', 'words', 'text']

Label counts
label
targeted_threat       65
nontargeted_threat    34
unrelated             37

Rows with no text: 0
Median words per article: 329


**Results**
- 136 labeled articles, with the three labels split 65 `targeted_threat`, 34 `nontargeted_threat` and 37 `unrelated`.
- The largest class is 48% of the set. A model that always guessed that one label would already score 48% accuracy without learning anything, which is why every result in this notebook is reported as Cohen's kappa instead.
- No article is missing its text, and the median article is 329 words, so the classifiers have real text to work with rather than headlines.
- The smallest class has 34 articles. Spread over five folds that is about seven per test fold, so per class results can show the direction of a difference but not a precise size.

## Block 2: Controls before any model runs

There is two things that would affect the comparison. The first is duplicated text. If the same article appears twice, one copy can land in the training part of a fold and the other in the test part, so the model gets scored on something it already learned. The score comes out higher than it should and nothing in the output looks wrong.

The second is the folds. CV splits the articles into five groups, trains on four and predicts the fifth, rotating until every article has been predicted exactly once by a model that never saw it. For a comparison between candidates to mean anything, they all have to be scored on the identical split, so the folds are built once here and reused by every candidate. Printing what is inside each fold also shows how many articles of the smallest class land in each test set, which sets the limit on how precisely per-class results can be read.

In [2]:
from sklearn.model_selection import StratifiedKFold

texts = training["text"].str.strip()

print("Duplicate full texts:", len(texts) - texts.nunique())
print("Duplicate URLs:", len(training) - training["url"].nunique())

# Wire copy is the realistic case: the same story republished by two outlets, so the
# bodies match for a while and then diverge.
print("Articles sharing their first 400 characters:", len(texts.str[:400]) - texts.str[:400].nunique())
print()

# Built once and reused by every candidate, so no model gets an easier split than another.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for number, (train_index, test_index) in enumerate(folds.split(training["text"], training["label"]), 1):
    counts = training["label"].iloc[test_index].value_counts().reindex(LABELS).fillna(0).astype(int)
    rows.append({"fold": number, "test articles": len(test_index), **counts.to_dict()})

pd.DataFrame(rows).set_index("fold")


Duplicate full texts: 0
Duplicate URLs: 0
Articles sharing their first 400 characters: 0



,test articles,targeted_threat,nontargeted_threat,unrelated
fold,,,,
1,28,13,7,8
2,27,13,7,7
3,27,13,7,7
4,27,13,7,7
5,27,13,6,8


**Results**

- No duplicated text, no duplicated URLs, and no article sharing its opening with another. Nothing can leak from a training fold into a test fold.
- The five folds are even: 27 or 28 test articles each, with 13 `targeted_threat`, 6 or 7 `nontargeted_threat` and 7 or 8 `unrelated`.
- Six or seven articles of the smallest class per test fold is the measurement floor. One article changing its prediction moves that class's recall by about 14 points, so per class numbers are read as direction, not as precise values.

## Block 3: The shared machinery, and the reference

Every candidate shares these parts, so the only difference between them is the model itself.

We turn the labels into numbers once, for all candidates. One candidate accepts only numeric labels, and encoding everything the same way avoids a special case that would treat it differently. Kappa is the same number either way.

We write the prediction as an explicit loop rather than one library call, so the notebook shows what happens: for each of the five folds a fresh model trains on four fifths of the articles and predicts the remaining fifth, until every article holds one prediction from a model that never trained on it. We compute the class balancing weights inside the loop, on the training part only, so a test fold never influences the balancing of its own prediction.

We end by running the reference once, to confirm the machinery produces a sensible number before all the candidates run over many splits.splits.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import cohen_kappa_score
from xgboost import XGBClassifier

encoder = LabelEncoder().fit(LABELS)
y = encoder.transform(training["label"])

def predict_out_of_fold(build, seed, features=None, labels=None):
    """Give every article one prediction, using a model that never trained on it.

    features defaults to the article text and labels to the three-class encoding.
    Candidates working from precomputed vectors, or from a different label scheme,
    pass their own.
    """
    X = training["text"].to_numpy() if features is None else features
    target = y if labels is None else labels
    split = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    predicted = np.empty(len(target), dtype=int)

    for train_index, test_index in split.split(X, target):
        model = build()
        weights = compute_sample_weight("balanced", target[train_index])
        model.fit(
            X[train_index], target[train_index],
            **{model.steps[-1][0] + "__sample_weight": weights},
        )
        predicted[test_index] = model.predict(X[test_index])

    return predicted
def words():
    """The word features, identical for every candidate that uses them."""
    return TfidfVectorizer(stop_words="english", min_df=2)


CANDIDATES = {
    "1 logistic regression": lambda: make_pipeline(
        words(), LogisticRegression(max_iter=1000)),
    # dual=True is the right solver when there are more word features than articles
    "2 linear SVM": lambda: make_pipeline(
        words(), LinearSVC(dual=True)),
    "3 XGBoost": lambda: make_pipeline(
        words(), XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.1, subsample=0.9,
            colsample_bytree=0.8, eval_metric="mlogloss", n_jobs=4, random_state=42)),
    # 100 components out of a few thousand word columns, so words that travel together
    # collapse into one direction instead of staying separate.
    "4 SVD + logistic regression": lambda: make_pipeline(
        words(), TruncatedSVD(n_components=100, random_state=42),
        LogisticRegression(max_iter=1000)),
}

reference = predict_out_of_fold(CANDIDATES["1 logistic regression"], seed=42)

print("Candidates defined:", len(CANDIDATES))
print("Reference kappa, one split:", round(cohen_kappa_score(y, reference), 3))

Candidates defined: 4
Reference kappa, one split: 0.733


**Results**

- Four candidates defined, and the reference produces a kappa of 0.733 on this split.
- The machinery runs end to end, so the comparison can proceed. Whether this reproduces the baseline from notebook 07 is checked in the next block, where the reference is averaged over ten splits rather than measured on one.

## Block 4: All candidates on the same ten splits

Each candidate runs the full five-fold CV ten times, with a different random fold split each time, and every candidate sees the same ten splits.

With 136 articles, a single cut can appear by chance, so the range across the ten matters more than one number. Two candidates whose ranges barely overlap are genuinely different. Two whose ranges overlap cannot be distinguished at this sample size, and reporting that is the honest result.

The per-split numbers are written to `08_model_results.csv`

In [4]:
results = {}

for name, build in CANDIDATES.items():
    kappas = [cohen_kappa_score(y, predict_out_of_fold(build, seed)) for seed in range(10)]
    results[name] = kappas
    print(f"{name:30} average kappa {np.mean(kappas):.3f}")

scores = pd.DataFrame(results).T
scores.columns = ["split " + str(seed) for seed in range(10)]
scores.round(4).to_csv(os.path.join(FOLDER, "08_model_results.csv"))

summary = pd.DataFrame({
    "lowest": scores.min(axis=1),
    "average": scores.mean(axis=1),
    "highest": scores.max(axis=1),
}).round(3)

print()
print("Saved: 08_model_results.csv")
summary

1 logistic regression          average kappa 0.676
2 linear SVM                   average kappa 0.662
3 XGBoost                      average kappa 0.529
4 SVD + logistic regression    average kappa 0.684

Saved: 08_model_results.csv


,lowest,average,highest
1 logistic regression,0.601,0.676,0.707
2 linear SVM,0.625,0.662,0.709
3 XGBoost,0.460,0.529,0.608
4 SVD + logistic regression,0.601,0.684,0.719


**Results**

- The reference averages 0.676, which reproduces the 0.68 reported in notebook 07. The machinery is sound, so the other three numbers can be read against it.
- The reference's own range across the ten splits runs 0.601 to 0.707, a spread of 0.106. That spread sets the resolution of this experiment: a difference smaller than it cannot be separated from the luck of the random split.
- Candidate 4 posts the highest average at 0.684, which is 0.008 above the reference. That is far below the spread, so by the decision rule written at the top of this notebook it does not count as a win. The same holds for candidate 2 at 0.662.
- Candidate 3 is the one real difference, and it is in the wrong direction. Its best split, 0.608, barely clears the reference's worst, 0.601, so the two ranges almost fail to overlap. Hundreds of trees over thousands of sparse word columns need more than 136 examples to find the word combinations they are built for.
- Three of the four candidates cannot be told apart and the one that differs is worse. What all four share is the representation: every one reads the articles as TF-IDF word counts. Swapping the classifier on top of those counts, or compressing them into topics first, does not move the result. That points at the representation rather than the classifier, which is the one thing candidate 5 changes.

## Block 5: The fifth candidate

Candidates 1 through 4 read each article as TF-IDF word counts. This one replaces that step with a pretrained network that returns one fixed length vector per article, and keeps the same logistic regression on top, so the comparison isolates the representation.

We load the model, embed the 136 articles, and measure two things: the shape of the result, and how many tokens of each article the model reads. The second matters because the model processes a fixed number of tokens and drops the rest, which decides whether candidate 5 works from the same text as the other four.ur.

In [5]:
#To avoid a Windows cache notice from the model download
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from sentence_transformers import SentenceTransformer

# A small pretrained model, about 90 MB, downloaded once and then read from cache.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(training["text"].tolist(), show_progress_bar=False)

print("Embedding shape:", embeddings.shape)
print()
print("Tokens the model reads per article:", embedder.max_seq_length)

# The tokenizer is the piece that cuts text into tokens, so it can say how long each
# article is in the units the model actually counts.
tokens = pd.Series([len(embedder.tokenizer.encode(text)) for text in training["text"]])
print("Median tokens per article:", int(tokens.median()))
print("Articles longer than the limit:", int((tokens > embedder.max_seq_length).sum()), "of", len(tokens))

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (270 > 256). Running this sequence through the model will result in indexing errors


Embedding shape: (136, 384)

Tokens the model reads per article: 256
Median tokens per article: 408
Articles longer than the limit: 96 of 136


**Results**

- Every article produced a vector of 384 numbers, so the representation step works on all 136.
- The model reads 256 tokens per article. The median article is 408 tokens and 96 of the 136 run longer than the limit, so for most of the set the vector is built from the opening of the article and the rest is dropped.
- Candidate 5 is therefore split in two before the comparison. 5a uses the vector as produced here, from the opening only. 5b splits each article into pieces that fit the limit, embeds each piece and averages them, so the whole article is represented. Running both answers whether the dropped text carries classification signal rather than assuming it either way.

## Block 6: The embedding candidates on the same ten splits

We compute the vectors once, outside the folds. The network producing them is pretrained and never sees the labels, so nothing about a test fold reaches the model through that step. The logistic regression on top is still fitted inside each fold, on the training part only, exactly as for the first four candidates.

Both candidates run through the same ten splits as the others, and the results table is rewritten with all six rows.

In [6]:
def chunks(text, size=180):
    """Pieces of about 180 words, which fits inside the model's 256 token limit."""
    words_in_text = text.split()
    pieces = [" ".join(words_in_text[i:i + size]) for i in range(0, len(words_in_text), size)]
    return pieces if pieces else [""]


averaged = np.vstack([
    embedder.encode(chunks(text), show_progress_bar=False).mean(axis=0)
    for text in training["text"]
])

print("Averaged embedding shape:", averaged.shape)
print()

# The representation is already numeric, so the pipeline is the classifier alone.
logistic_only = lambda: make_pipeline(LogisticRegression(max_iter=1000))

for name, features in [("5a embeddings, opening only", embeddings),
                       ("5b embeddings, whole article", averaged)]:
    kappas = [cohen_kappa_score(y, predict_out_of_fold(logistic_only, seed, features))
              for seed in range(10)]
    results[name] = kappas
    print(f"{name:30} average kappa {np.mean(kappas):.3f}")

scores = pd.DataFrame(results).T
scores.columns = ["split " + str(seed) for seed in range(10)]
scores.round(4).to_csv(os.path.join(FOLDER, "08_model_results.csv"))

summary = pd.DataFrame({
    "lowest": scores.min(axis=1),
    "average": scores.mean(axis=1),
    "highest": scores.max(axis=1),
}).round(3)

print()
print("Saved: 08_model_results.csv")
summary

Averaged embedding shape: (136, 384)

5a embeddings, opening only    average kappa 0.635
5b embeddings, whole article   average kappa 0.577

Saved: 08_model_results.csv


,lowest,average,highest
1 logistic regression,0.601,0.676,0.707
2 linear SVM,0.625,0.662,0.709
3 XGBoost,0.460,0.529,0.608
4 SVD + logistic regression,0.601,0.684,0.719
"5a embeddings, opening only",0.590,0.635,0.661
"5b embeddings, whole article",0.521,0.577,0.628


**Results**

- No candidate beats the reference. Candidate 4 holds the highest average at 0.684, which is 0.008 above it and far inside the 0.106 spread the reference shows across the ten splits, so by the decision rule it is not a win.
- The embedding candidates land below the reference, 5a at 0.635 and 5b at 0.577. Their ranges still overlap the reference's, so neither is a measurable loss, but neither is an improvement either.
- 5b sits 0.058 below 5a. Reading the whole article made the representation worse than reading only its opening. News articles carry their substance in the first paragraphs, while the tail holds boilerplate, navigation text and summaries of unrelated stories, and averaging that in dilutes the vector. This is the question block 5 set out to answer, and the answer is that the dropped text was not carrying signal worth recovering.
- Candidate 3 remains the only clear difference, and it is in the wrong direction.

**What the table says taken together.** Five alternatives were tried: a different linear classifier, a tree ensemble, a compression of the word features, and a pretrained network reading the text two different ways. None beats TF-IDF with logistic regression at this sample size. Changing the classifier does not help and changing the representation does not help, which points at the 136 labeled articles as the binding constraint rather than the choice of model. A second explanation for the embedding result, not tested here, is that a small general purpose sentence model is poorly matched to this domain.

## Block 7: Where the model's errors sit

**Relationship between annotator agreement and model error.**

The labeled set records, for each article, whether the annotators agreed or whether the label had to be adjudicated. The reference model runs over the same ten splits used throughout this notebook, and every article is scored on how many of those ten times it was classified correctly, so an article's hit rate does not depend on which fold it happened to land in. Those hit rates are then grouped by the agreement column.

A model cannot be more consistent than the labels it learns from. If the errors concentrate in the articles the annotators disputed, the limit on every candidate in this notebook is the labels rather than the model.

In [7]:
# Each article is predicted ten times, once per split, so its hit rate is stable rather
# than depending on which single fold it happened to fall in.
hits = np.zeros(len(y))
for seed in range(10):
    hits += predict_out_of_fold(CANDIDATES["1 logistic regression"], seed) == y

print("Values in the agreement column")
print(training["agreement"].value_counts(dropna=False).to_string())
print()

by_agreement = pd.DataFrame({"agreement": training["agreement"], "hits": hits})

table = by_agreement.groupby("agreement", dropna=False).agg(
    articles=("hits", "size"),
    correct=("hits", "mean"),
)
table["% correct"] = (table["correct"] / 10 * 100).round(1)

print(table[["articles", "% correct"]].to_string())
print()
print("Label mix inside each agreement group, in percent")

# Accuracy moves with the class mix, so the mix is reported next to it.
mix = pd.crosstab(training["agreement"], training["label"], normalize="index")
(mix[LABELS] * 100).round(1)

Values in the agreement column
agreement
all three agreed    72
adjudicated         41
two of three        23

                  articles  % correct
agreement                            
adjudicated             41       75.9
all three agreed        72       85.4
two of three            23       68.3

Label mix inside each agreement group, in percent


label,targeted_threat,nontargeted_threat,unrelated
agreement,,,
adjudicated,46.3,31.7,22.0
all three agreed,45.8,16.7,37.5
two of three,56.5,39.1,4.3


**Results**

- The class mix is not constant across the three groups, so the accuracy column cannot be read as a measure of label noise on its own. The share of `unrelated` in each group runs 37.5, 22.0 and 4.3 percent, and the accuracy runs 85.4, 75.9 and 68.3 in the same order. Most of the accuracy gap is explained by how much of the easiest class each group happens to contain.
- The mix itself is the finding. `nontargeted_threat` is 16.7 percent of the unanimous articles and 39.1 percent of the two of three articles. `unrelated` is 37.5 percent of the unanimous articles and 4.3 percent of the two of three articles. Annotators agreed readily on what was not a threat, and the articles they could not agree on are concentrated in one class.
- 64 of the 136 articles, almost half the training set, carry a label the annotators did not reach unanimously, and those articles cluster in `nontargeted_threat`.
- Notebook 06 reported a Fleiss kappa of 0.575 among three annotators on 100 articles, and the reference model reaches 0.676 against the adjudicated labels. The two come from different statistics on different articles, so they are not directly comparable, but both sit in the same range, and no classifier can be more consistent than the labels it learns from.
- What this does not establish is which class `nontargeted_threat` was being confused with.

## Block 8: The measurement against shuffled labels

The reference runs again over the same ten splits, but with the labels shuffled at random, so no relationship between an article's text and its label survives.

A pipeline scored this way should land near zero, since zero is what kappa reports for agreement no better than chance. A result meaningfully above zero would mean the procedure can manufacture agreement out of noise, and every number in this notebook would have to be discarded.

In [8]:
shuffled_kappas = []

for seed in range(10):
    # A fresh shuffle per split, so the control is averaged the same way the real
    # results are.
    shuffled = np.random.default_rng(seed).permutation(y)
    predicted = predict_out_of_fold(
        CANDIDATES["1 logistic regression"], seed, labels=shuffled)
    shuffled_kappas.append(cohen_kappa_score(shuffled, predicted))

print("Reference on the real labels:", round(scores.loc["1 logistic regression"].mean(), 3))
print("Reference on shuffled labels:", round(np.mean(shuffled_kappas), 3))
print("Shuffled range:", round(min(shuffled_kappas), 3), "to", round(max(shuffled_kappas), 3))

Reference on the real labels: 0.676
Reference on shuffled labels: -0.041
Shuffled range: -0.14 to 0.069


**Results**

- On shuffled labels the reference averages a kappa of -0.041, across a range of -0.14 to 0.069. On the real labels it averages 0.676.
- Kappa reports zero for agreement no better than chance, and the shuffled runs sit on zero, drifting to either side as expected from random cuts of a small set. The procedure does not produce agreement where none exists.

## Conclusions

| Model | average kappa |
|---|---|
| TF-IDF + logistic regression (reference) | 0.676 |
| TF-IDF + linear SVM | 0.662 |
| TF-IDF + XGBoost | 0.529 |
| TF-IDF + SVD + logistic regression | 0.684 |
| Embeddings, opening only | 0.635 |
| Embeddings, whole article | 0.577 |
| Reference on shuffled labels (control) | -0.041 |

**TF-IDF with logistic regression stays as the model of record.** No candidate beat it. The best average, 0.684, sits 0.008 above the reference, well inside the 0.106 the reference moves on the cut of the folds alone.

**Limits**

- 136 labeled articles, all from the 2021 case.
- One reasonable setting per candidate, no search over settings.
- An improvement smaller than about 0.05 kappa is invisible at this sample size.

**Raised for approval, not decided here.** Merging `targeted_threat` and `nontargeted_threat` would remove the distinction block 7 showed the annotators could not make reliably, at the cost of separating a threat against a named school from a general one.

**Next steps**

- #TODO: Redo Questions 4 and 5 counting only the articles classified as threats.
- #TODO: Use SHAP with a text masker on the reference model for Question 2.